# Homework 5: Model Evaluation and Hyperparameter Tuning

This notebook performs model evaluation and hyperparameter tuning on the Breast Cancer Wisconsin dataset using various techniques including:
- Pipeline creation with cross-validation
- Learning curves analysis
- Validation curves
- GridSearchCV for exhaustive hyperparameter search
- RandomizedSearchCV for randomized hyperparameter search
- Class imbalance handling through resampling
- ROC curve comparison

## Overview

This notebook demonstrates the complete machine learning workflow for model evaluation and hyperparameter tuning:

| Task | Goal | Method |
|------|------|--------|
| **Task 1** | Establish baseline model performance | Pipeline + Cross-Validation |
| **Task 2A** | Diagnose bias-variance trade-off | Learning Curves |
| **Task 2B** | Find optimal PCA dimensions | Validation Curves |
| **Task 3A** | Exhaustive hyperparameter search | GridSearchCV (72 combinations) |
| **Task 3B** | Efficient hyperparameter search | RandomizedSearchCV (30 samples) |
| **Task 4** | Handle class imbalance | Downsampling + Upsampling |
| **Task 5** | Compare model discrimination ability | ROC Curves |

Learn how different techniques improve model performance and address real-world challenges like class imbalance and hyperparameter optimization.

## Import Required Libraries

This section imports all necessary libraries from scikit-learn, scipy, and standard Python data science packages:

- **scikit-learn**: Tools for data loading, preprocessing, model selection, and evaluation
- **scipy.stats.loguniform**: A log-uniform distribution for sampling hyperparameters across multiple orders of magnitude
- **matplotlib**: For creating visualizations and plots
- **numpy and pandas**: For numerical computations and data manipulation
- **%matplotlib inline**: Display plots directly in the notebook

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.stats import loguniform
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score, learning_curve, train_test_split, validation_curve)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

%matplotlib inline

## Setup Output Directory and Load Dataset

The **Breast Cancer Wisconsin dataset** is a classic dataset for binary classification in machine learning:
- **569 samples** with **30 features** extracted from digitized images of breast mass
- **2 classes**: Malignant (cancer) and Benign (non-cancer)
- Features represent cell characteristics like radius, texture, perimeter, area, smoothness, etc.

An output folder is created to save all results (plots, CSV files) for analysis.

In [ ]:
# Create output folder for results
output_folder = "model_evaluation_results"
os.makedirs(output_folder, exist_ok=True)

random_state = 1

# 1. Load and prepare the Breast Cancer Wisconsin dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

print("Breast Cancer Wisconsin dataset")
print(f"Samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Class names: {list(cancer.target_names)}")
print(f"Full class distribution: {np.bincount(y)}")

## Train-Test Split and Cross-Validation Setup

**Key concepts:**
- **Train-Test Split**: Data is split into 80% training and 20% testing to evaluate model performance on unseen data
- **Stratified Split**: Ensures both training and test sets have similar class distributions (important for imbalanced datasets)
- **StratifiedKFold**: Creates 10 stratified folds for cross-validation, which estimates model performance by training on 9 folds and validating on 1, repeated 10 times
- **Random State**: Ensures reproducibility of results

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=random_state,
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Training class distribution: {np.bincount(y_train)}")
print(f"Testing class distribution: {np.bincount(y_test)}")

# Use the same stratified folds throughout the assignment
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=random_state,
)

## Task 1: Pipeline and Cross-Validation

**Objective**: Build a baseline model using a pipeline and evaluate it with cross-validation.

**Pipeline Components**:
1. **StandardScaler**: Normalizes features to have zero mean and unit variance (essential for distance-based algorithms)
2. **PCA (Principal Component Analysis)**: Reduces 30 features to 2 principal components while retaining most variance
3. **LogisticRegression**: A linear classifier suitable for binary classification

**Evaluation**:
- Cross-validation scores show model robustness across different data splits
- Baseline test accuracy provides a reference point for comparing with optimized models

In [ ]:
# 2. Task 1: Pipeline and cross-validation
pipe_lr = make_pipeline(
    StandardScaler(),
    PCA(n_components=2),
    LogisticRegression(max_iter=10000, random_state=random_state),
)

cv_scores = cross_val_score(
    estimator=pipe_lr,
    X=X_train,
    y=y_train,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
)

pipe_lr.fit(X_train, y_train)
baseline_test_accuracy = pipe_lr.score(X_test, y_test)

print("Task 1: Pipeline and cross-validation")
print(f"Cross-validation scores: {np.round(cv_scores, 3)}")
print(f"Mean CV accuracy: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
print(f"Baseline test accuracy: {baseline_test_accuracy:.3f}")

## Task 2A: Learning Curve

**What is a Learning Curve?**
A learning curve plots training and validation accuracy against the number of training samples. It helps identify:
- **High Bias (Underfitting)**: Both curves plateau at low accuracy → model is too simple
- **High Variance (Overfitting)**: Large gap between training and validation curves → model memorizes training data
- **Good Fit**: Curves close together and high accuracy → model generalizes well

**Interpretation**: If the validation curve plateaus below training curve, collecting more data may help improve performance.

In [ ]:
# 3. Task 2A: Learning curve
train_sizes, train_scores, validation_scores = learning_curve(
    estimator=pipe_lr,
    X=X_train,
    y=y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
validation_mean = np.mean(validation_scores, axis=1)
validation_std = np.std(validation_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, marker="o", label="Training accuracy")
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15)

plt.plot(train_sizes, validation_mean, marker="s", linestyle="--", label="Validation accuracy")
plt.fill_between(train_sizes, validation_mean - validation_std, validation_mean + validation_std, alpha=0.15)

plt.xlabel("Number of training examples")
plt.ylabel("Accuracy")
plt.title("Learning Curve: PCA and Logistic Regression")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig(os.path.join(output_folder, "learning_curve.png"))
plt.show()

## Task 2B: Validation Curve for PCA Components

**What is a Validation Curve?**
A validation curve plots training and validation accuracy against a single hyperparameter value. It helps:
- Identify the optimal value for that hyperparameter
- Understand how the hyperparameter affects model performance

**For PCA Components**: 
- With too few components, we lose important information (underfitting)
- With too many components, we include noise and computation increases (potential overfitting)
- The optimal point balances information retention with model simplicity

In [ ]:
# 4. Task 2B: Validation curve for the number of PCA components
pca_components = [2, 5, 10, 15, 20, 25, 30]

validation_train_scores, validation_test_scores = validation_curve(
    estimator=pipe_lr,
    X=X_train,
    y=y_train,
    param_name="pca__n_components",
    param_range=pca_components,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
)

validation_train_mean = np.mean(validation_train_scores, axis=1)
validation_train_std = np.std(validation_train_scores, axis=1)
validation_test_mean = np.mean(validation_test_scores, axis=1)
validation_test_std = np.std(validation_test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(pca_components, validation_train_mean, marker="o", label="Training accuracy")
plt.fill_between(pca_components, validation_train_mean - validation_train_std, validation_train_mean + validation_train_std, alpha=0.15)

plt.plot(pca_components, validation_test_mean, marker="s", linestyle="--", label="Validation accuracy")
plt.fill_between(pca_components, validation_test_mean - validation_test_std, validation_test_mean + validation_test_std, alpha=0.15)

plt.xlabel("Number of PCA components")
plt.ylabel("Accuracy")
plt.title("Validation Curve: Number of PCA Components")
plt.xticks(pca_components)
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig(os.path.join(output_folder, "validation_curve_pca.png"))
plt.show()

best_pca_index = int(np.argmax(validation_test_mean))
best_pca_components = pca_components[best_pca_index]

print("Task 2: Learning and validation curves")
print("Saved learning_curve.png")
print("Saved validation_curve_pca.png")
print(f"Best PCA component count from the validation curve: {best_pca_components}")
print(f"Best mean validation accuracy: {validation_test_mean[best_pca_index]:.3f}")

## Task 3A: GridSearchCV for Hyperparameter Tuning

**What is GridSearchCV?**
GridSearchCV performs an **exhaustive search** over specified hyperparameter values:
- Tests every combination of parameters from the parameter grid
- Uses cross-validation to evaluate each combination
- Returns the best parameters and best score

**Trade-offs**:
- ✓ Thorough exploration of parameter space
- ✓ Guarantees finding the best combination within the grid
- ✗ Computationally expensive (tests all combinations: 6 × 2 × 6 = 72 combinations × 10-fold CV)

**Hyperparameters tuned**:
- `n_components`: Number of PCA dimensions (5-30)
- `solver`: Algorithm for LogisticRegression (liblinear, lbfgs)
- `C`: Inverse regularization strength (0.001-100)

In [ ]:
# 5. Task 3A: GridSearchCV for Logistic Regression
search_pipeline = make_pipeline(
    StandardScaler(),
    PCA(),
    LogisticRegression(max_iter=10000, random_state=random_state),
)

grid_parameters = {
    "pca__n_components": [5, 10, 15, 20, 25, 30],
    "logisticregression__solver": ["liblinear", "lbfgs"],
    "logisticregression__C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
}

grid_search = GridSearchCV(
    estimator=search_pipeline,
    param_grid=grid_parameters,
    scoring="accuracy",
    cv=cv,
    refit=True,
    n_jobs=-1,
)

print("Running GridSearchCV (this may take a few minutes)...")
grid_search.fit(X_train, y_train)
grid_test_accuracy = grid_search.score(X_test, y_test)

print("\nTask 3A: GridSearchCV")
print(f"Best CV accuracy: {grid_search.best_score_:.3f}")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Grid search test accuracy: {grid_test_accuracy:.3f}")

## Task 3B: RandomizedSearchCV for Hyperparameter Tuning

**What is RandomizedSearchCV?**
RandomizedSearchCV performs a **random search** over specified hyperparameter distributions:
- Randomly samples a fixed number of combinations from the parameter space
- Uses cross-validation to evaluate each sample
- Particularly useful for continuous distributions

**Trade-offs**:
- ✗ May not find the global best parameters
- ✓ Much faster than GridSearchCV (samples 30 combinations instead of 72)
- ✓ Can explore continuous parameter distributions efficiently

**Key difference**: 
- `C` uses `loguniform(0.001, 100)` - a continuous log-uniform distribution
- This better captures the exponential nature of regularization strength

In [ ]:
# 6. Task 3B: RandomizedSearchCV for comparison
random_parameters = {
    "pca__n_components": [5, 10, 15, 20, 25, 30],
    "logisticregression__solver": ["liblinear", "lbfgs"],
    "logisticregression__C": loguniform(0.001, 100.0),
}

random_search = RandomizedSearchCV(
    estimator=search_pipeline,
    param_distributions=random_parameters,
    n_iter=30,
    scoring="accuracy",
    cv=cv,
    refit=True,
    random_state=random_state,
    n_jobs=-1,
)

print("Running RandomizedSearchCV (this may take a few minutes)...")
random_search.fit(X_train, y_train)
random_test_accuracy = random_search.score(X_test, y_test)

print("\nTask 3B: RandomizedSearchCV")
print(f"Best CV accuracy: {random_search.best_score_:.3f}")
print(f"Best parameters: {random_search.best_params_}")
print(f"Randomized search test accuracy: {random_test_accuracy:.3f}")

## Task 4: Class Imbalance and Resampling

**What is Class Imbalance?**
Many real-world datasets have unequal class distributions. The Breast Cancer dataset has more benign cases than malignant, which can bias models to favor the majority class.

**Resampling Strategies**:
1. **Downsampling**: Reduce majority class size to match minority class
   - Pro: Faster training
   - Con: Lose information from majority class

2. **Upsampling**: Increase minority class size to match majority class (with replacement)
   - Pro: Keep all data
   - Con: Risk of overfitting due to duplicated samples

3. **Original**: Keep the original imbalanced distribution
   - Baseline for comparison

**Evaluation Metrics for Imbalanced Data**:
- **Recall per class**: Captures the model's ability to find each class
- **Macro F1**: Average F1-score across classes (equal weight to both classes)
- **Confusion matrix**: Shows false positives and false negatives per class

In [ ]:
# 7. Task 4: Compare original, downsampled, and upsampled training data
# Only the training set is resampled. The test set stays unchanged.
train_data = np.column_stack((X_train, y_train))
class_0 = train_data[train_data[:, -1] == 0]
class_1 = train_data[train_data[:, -1] == 1]

if len(class_0) < len(class_1):
    minority_class = class_0
    majority_class = class_1
else:
    minority_class = class_1
    majority_class = class_0

print(f"Minority class size: {len(minority_class)}")
print(f"Majority class size: {len(majority_class)}")

# Downsample the majority class to the minority class size.
majority_downsampled = resample(
    majority_class,
    replace=False,
    n_samples=len(minority_class),
    random_state=random_state,
)

downsampled_data = np.vstack((minority_class, majority_downsampled))
rng = np.random.default_rng(random_state)
rng.shuffle(downsampled_data)

X_train_downsampled = downsampled_data[:, :-1]
y_train_downsampled = downsampled_data[:, -1].astype(int)

# Upsample the minority class to the majority class size.
minority_upsampled = resample(
    minority_class,
    replace=True,
    n_samples=len(majority_class),
    random_state=random_state,
)

upsampled_data = np.vstack((majority_class, minority_upsampled))
rng.shuffle(upsampled_data)

X_train_upsampled = upsampled_data[:, :-1]
y_train_upsampled = upsampled_data[:, -1].astype(int)

print(f"Downsampled training set size: {len(X_train_downsampled)}")
print(f"Upsampled training set size: {len(X_train_upsampled)}")

## Evaluate Models on Different Resampling Strategies

In [ ]:
def evaluate_resampled_model(name, X_training, y_training):
    model = grid_search.best_estimator_
    model.fit(X_training, y_training)
    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    report = classification_report(
        y_test,
        predictions,
        target_names=cancer.target_names,
        output_dict=True,
        zero_division=0,
    )

    return {
        "Training data": name,
        "Training samples": len(y_training),
        "Class 0 samples": int(np.sum(y_training == 0)),
        "Class 1 samples": int(np.sum(y_training == 1)),
        "Accuracy": accuracy,
        "Malignant recall": report["malignant"]["recall"],
        "Benign recall": report["benign"]["recall"],
        "Macro F1": report["macro avg"]["f1-score"],
        "Confusion matrix": confusion_matrix(y_test, predictions).tolist(),
    }

print("Evaluating models on different resampling strategies...")
resampling_results = [
    evaluate_resampled_model("Original", X_train, y_train),
    evaluate_resampled_model(
        "Majority downsampled",
        X_train_downsampled,
        y_train_downsampled,
    ),
    evaluate_resampled_model(
        "Minority upsampled",
        X_train_upsampled,
        y_train_upsampled,
    ),
]

results_table = pd.DataFrame(resampling_results)
results_table.to_csv(
    os.path.join(output_folder, "resampling_comparison.csv"),
    index=False,
)

print("\nTask 4: Class imbalance and resampling")
print(
    results_table[
        [
            "Training data",
            "Training samples",
            "Class 0 samples",
            "Class 1 samples",
            "Accuracy",
            "Malignant recall",
            "Benign recall",
            "Macro F1",
        ]
    ].round(3).to_string(index=False)
)

for result in resampling_results:
    print(f"{result['Training data']} confusion matrix: {result['Confusion matrix']}")

## Task 5: ROC Curves Comparison

**What is an ROC Curve?**
ROC (Receiver Operating Characteristic) curves visualize the trade-off between True Positive Rate (TPR) and False Positive Rate (FPR) at different classification thresholds.

**Key Metrics**:
- **True Positive Rate (TPR/Recall)**: Proportion of actual positives correctly identified
- **False Positive Rate (FPR)**: Proportion of actual negatives incorrectly classified as positive
- **AUC (Area Under the Curve)**: Aggregates performance across all thresholds (0.5 = random, 1.0 = perfect)

**Interpretation**:
- A curve closer to the top-left corner indicates better performance
- Steeper curves show better discrimination between classes
- Comparing AUC values helps evaluate which resampling strategy performs best

**In this notebook**: We plot ROC curves for all three resampling strategies to see how class balance affects model discrimination ability.

In [ ]:
# 8. ROC curves for original and resampled models

roc_datasets = [
    ("Original", X_train, y_train),
    ("Majority downsampled", X_train_downsampled, y_train_downsampled),
    ("Minority upsampled", X_train_upsampled, y_train_upsampled),
]

plt.figure(figsize=(10, 8))

for name, X_training, y_training in roc_datasets:
    model = grid_search.best_estimator_

    model.fit(X_training, y_training)

    # Class 0 is malignant
    probabilities = model.predict_proba(X_test)[:, 0]

    fpr, tpr, thresholds = roc_curve(y_test, probabilities, pos_label=0)

    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})")


plt.plot([0, 1], [0, 1], linestyle="--", label="Random guessing (AUC = 0.500)")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves: Resampling Comparison")

plt.legend(loc="lower right")
plt.grid()
plt.tight_layout()

plt.savefig(os.path.join(output_folder, "roc_curves_resampling.png"))
plt.show()

print("Saved roc_curves_resampling.png")

## Summary

## Key Takeaways

### Model Evaluation Techniques
1. **Cross-Validation**: Provides more reliable performance estimates than a single train-test split
2. **Learning Curves**: Help diagnose whether a model suffers from high bias or high variance
3. **Validation Curves**: Guide hyperparameter selection by showing their impact on performance

### Hyperparameter Tuning
- **GridSearchCV**: Best when you have a small parameter space; guarantees finding the best combination
- **RandomizedSearchCV**: Better for large or continuous parameter spaces; faster but may miss global optimum
- **Loguniform Distribution**: Appropriate for regularization parameters that span multiple orders of magnitude

### Handling Class Imbalance
- **Downsampling**: Balances classes but loses information
- **Upsampling**: Preserves data but creates artificial duplicates
- **Evaluation Metrics**: Use Macro F1 and per-class recall to evaluate performance fairly on imbalanced data

### Performance Comparison
- Models trained on different resampling strategies show different AUC values in ROC curves
- The best strategy depends on your application priorities (sensitivity vs specificity)
- Generated files (`learning_curve.png`, `validation_curve_pca.png`, `resampling_comparison.csv`, `roc_curves_resampling.png`) provide detailed analysis

In [ ]:
# 9. Save a short text summary
best_resampling_result = max(
    resampling_results,
    key=lambda result: result["Macro F1"])

print(
    f"Best resampling strategy based on Macro F1: "
    f"{best_resampling_result['Training data']} "
    f"(Macro F1: {best_resampling_result['Macro F1']:.3f})"
)
print(f"\nAll results were saved in: {output_folder}")
print(f"\nGenerated files:")
print(f"  - learning_curve.png")
print(f"  - validation_curve_pca.png")
print(f"  - resampling_comparison.csv")
print(f"  - roc_curves_resampling.png")